# Sistema de Reconhecimento Facial — Notebook
Este notebook permite coletar faces via upload, treinar o modelo (LBPH ou CNN/Transfer Learning) e reconhecer faces em uma imagem.

Todo o código da aplicação está embutido neste notebook (não depende de `face_recognition_app.py`).

In [1]:
from pathlib import Path
import os, cv2, numpy as np, pickle, urllib.request
from ipywidgets import FileUpload, Text, Button, Dropdown, VBox, HBox
from IPython.display import display

try:
    from tensorflow import keras
except Exception:
    keras = None

ROOT_DIR = Path.cwd()
DATASET_DIR = ROOT_DIR / 'dataset'
TRAINER_DIR = ROOT_DIR / 'trainer'
MODEL_CACHE_DIR = ROOT_DIR / 'assets' / 'models'
YUNET_MODEL_PATH = MODEL_CACHE_DIR / 'face_detection_yunet_2023mar.onnx'
YUNET_MODEL_URL = (
    'https://raw.githubusercontent.com/opencv/opencv_zoo/main/models/face_detection_yunet/'
    'face_detection_yunet_2023mar.onnx'
)

_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
_YUNET = None


def ensure_dirs() -> None:
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    TRAINER_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _download_yunet_model_if_needed() -> None:
    if YUNET_MODEL_PATH.exists():
        return
    ensure_dirs()
    urllib.request.urlretrieve(YUNET_MODEL_URL, str(YUNET_MODEL_PATH))


def _get_yunet_detector(width: int, height: int):
    global _YUNET
    _download_yunet_model_if_needed()

    score_threshold = float(os.environ.get('YUNET_SCORE_THRESHOLD', '0.88'))
    nms_threshold = float(os.environ.get('YUNET_NMS_THRESHOLD', '0.3'))
    top_k = int(os.environ.get('YUNET_TOP_K', '5000'))

    if _YUNET is None:
        _YUNET = cv2.FaceDetectorYN.create(
            str(YUNET_MODEL_PATH),
            '',
            (width, height),
            score_threshold,
            nms_threshold,
            top_k,
        )
    else:
        _YUNET.setInputSize((width, height))
    return _YUNET


def detect_faces(image: np.ndarray, detector: str | None = None):
    if image is None or image.size == 0:
        return []

    if detector is None:
        detector = os.environ.get('FACE_DETECTOR', 'haar').strip().lower()

    if image.ndim == 2:
        gray = image
        bgr = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        bgr = image
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if detector == 'yunet':
        try:
            h, w = bgr.shape[:2]
            yunet = _get_yunet_detector(w, h)
            _, faces = yunet.detect(bgr)
            if faces is None:
                return []
            boxes = []
            for row in faces:
                x, y, bw, bh = row[:4].astype(int).tolist()
                x = max(0, x)
                y = max(0, y)
                bw = max(1, min(bw, w - x))
                bh = max(1, min(bh, h - y))
                boxes.append((x, y, bw, bh))
            return boxes
        except Exception:
            pass

    faces = _CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48))
    return [(int(x), int(y), int(w), int(h)) for (x, y, w, h) in faces]


def _load_dataset(image_size=(200, 200)):
    X, y = [], []
    label_ids = {}
    current_id = 0

    for person_dir in sorted(DATASET_DIR.glob('*')):
        if not person_dir.is_dir():
            continue
        person = person_dir.name
        if person not in label_ids:
            label_ids[person] = current_id
            current_id += 1

        for img_path in person_dir.glob('*.jpg'):
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            resized = cv2.resize(img, image_size)
            X.append(resized)
            y.append(label_ids[person])

    return np.array(X), np.array(y), label_ids


def _save_labels(label_ids):
    ensure_dirs()
    with open(TRAINER_DIR / 'labels.pickle', 'wb') as f:
        pickle.dump(label_ids, f)


def _build_small_cnn(num_classes: int):
    model = keras.Sequential(
        [
            keras.layers.Input(shape=(200, 200, 1)),
            keras.layers.Conv2D(32, 3, activation='relu'),
            keras.layers.MaxPool2D(),
            keras.layers.Conv2D(64, 3, activation='relu'),
            keras.layers.MaxPool2D(),
            keras.layers.Conv2D(128, 3, activation='relu'),
            keras.layers.GlobalAveragePooling2D(),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(num_classes, activation='softmax'),
        ]
    )
    return model


def _build_transfer_model(num_classes: int):
    base = keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(200, 200, 3),
    )
    base.trainable = False

    inputs = keras.Input(shape=(200, 200, 3))
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.2)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    return model


def train_recognizer(method: str = 'lbph') -> bool:
    ensure_dirs()

    X, y, label_ids = _load_dataset((200, 200))
    if len(X) == 0 or len(label_ids) < 2:
        print('Dataset insuficiente. Inclua ao menos 2 pessoas com imagens válidas.')
        return False

    _save_labels(label_ids)
    method = (method or 'lbph').strip().lower()

    if method == 'lbph':
        if not hasattr(cv2, 'face'):
            print('cv2.face indisponível. Instale opencv-contrib-python.')
            return False
        recognizer = cv2.face.LBPHFaceRecognizer_create()
        recognizer.train(X, y)
        recognizer.save(str(TRAINER_DIR / 'trainer.yml'))
        return True

    if keras is None:
        print('TensorFlow indisponível para métodos baseados em rede neural.')
        return False

    if method in ('cnn', 'transfer_yunet'):
        if method == 'transfer_yunet':
            X_net = np.stack([cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) for img in X]).astype('float32')
            model = _build_transfer_model(num_classes=len(label_ids))
            epochs = int(os.environ.get('FACE_TL_EPOCHS', '8'))
            batch_size = int(os.environ.get('FACE_TL_BATCH', '16'))
        else:
            X_net = (X.astype('float32') / 255.0)[..., None]
            model = _build_small_cnn(num_classes=len(label_ids))
            epochs = int(os.environ.get('FACE_CNN_EPOCHS', '12'))
            batch_size = int(os.environ.get('FACE_CNN_BATCH', '16'))

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )
        model.fit(X_net, y, epochs=epochs, batch_size=batch_size, verbose=1)
        model.save(str(TRAINER_DIR / 'model_tf.h5'))
        return True

    print(f'Método desconhecido: {method}')
    return False


ensure_dirs()
detector = os.environ.get('FACE_DETECTOR', 'yunet').strip().lower()
if detector not in ('yunet', 'haar'):
    detector = 'yunet'
os.environ['FACE_DETECTOR'] = detector
print('Detector facial ativo:', detector)
if detector == 'yunet':
    print('YuNet ativo (detector pré-treinado com transfer learning para detecção).')

Detector facial ativo: yunet
YuNet ativo (detector pré-treinado com transfer learning para detecção).


## Coleta por upload
Carregue imagens e salve faces recortadas em `dataset/<nome>/`.

In [2]:
name = Text(description='Nome')
u = FileUpload(accept='.jpg,.jpeg,.png', multiple=True)
btn = Button(description='Salvar faces')
box = VBox([name, u, btn])
def on_click(_):
    if not name.value:
        print('Informe um nome'); return
    person_dir = Path('dataset')/name.value
    person_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for item in u.value.values():
        data = np.frombuffer(item['content'], dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = detect_faces(gray)
        for (x,y,w,h) in faces:
            face = cv2.resize(gray[y:y+h, x:x+w], (200,200))
            next_i = len(list(person_dir.glob('*.jpg')))
            cv2.imwrite(str(person_dir/f'{next_i:03d}.jpg'), face)
            count += 1
    print(f'Faces salvas: {count} em {person_dir}')
btn.on_click(on_click)
display(box)

## Treinamento
Escolha o método e treine o modelo.
- `lbph`: baseline clássico com OpenCV.
- `cnn`: CNN pequena treinada do zero.
- `transfer_yunet`: Transfer learning com MobileNetV2 para reconhecimento, usando faces detectadas com YuNet.

In [3]:
method = Dropdown(description='Método', options=['lbph','cnn','transfer_yunet'], value='transfer_yunet')
train_btn = Button(description='Treinar')
train_box = HBox([method, train_btn])
def train_click(_):
    ok = train_recognizer(method.value)
    print('Treinado:', ok)
train_btn.on_click(train_click)
display(train_box)

## Predição por upload
Envie uma imagem para reconhecer e visualize o resultado.

In [4]:
import matplotlib.pyplot as plt

labels_path = Path('trainer') / 'labels.pickle'
model_tf_path = Path('trainer') / 'model_tf.h5'
model_lbph_path = Path('trainer') / 'trainer.yml'

upload_pred = FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
pred_btn = Button(description='Reconhecer')
display(VBox([upload_pred, pred_btn]))

def predict_upload(_):
    if not labels_path.exists() or (not model_tf_path.exists() and not model_lbph_path.exists()):
        print('Modelo não encontrado')
        return

    if not upload_pred.value:
        print('Envie uma imagem')
        return

    with open(labels_path, 'rb') as f:
        label_ids = pickle.load(f)
    id_labels = {v: k for k, v in label_ids.items()}

    use_tf = model_tf_path.exists()
    recognizer = None
    model = None
    expected_channels = 1

    if use_tf:
        if keras is None:
            print('TensorFlow indisponível. Use LBPH ou instale TensorFlow para predição com model_tf.h5.')
            return
        model = keras.models.load_model(str(model_tf_path))
        expected_channels = int(model.input_shape[-1]) if model.input_shape[-1] is not None else 1
    else:
        if not hasattr(cv2, 'face'):
            print('cv2.face indisponível. Instale opencv-contrib-python para usar LBPH.')
            return
        recognizer = cv2.face.LBPHFaceRecognizer_create()
        recognizer.read(str(model_lbph_path))

    item = list(upload_pred.value.values())[0]
    data = np.frombuffer(item['content'], dtype=np.uint8)
    image = cv2.imdecode(data, cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = detect_faces(gray)

    for (x, y, w, h) in faces:
        face = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        if use_tf:
            if expected_channels == 3:
                rgb_face = cv2.cvtColor(face, cv2.COLOR_GRAY2RGB).astype('float32')
                arr = rgb_face[None, ...]
            else:
                arr = (face.astype('float32') / 255.0)[None, ..., None]
            probs = model.predict(arr, verbose=0)[0]
            lid = int(np.argmax(probs))
            conf = float(np.max(probs) * 100.0)
            name = id_labels.get(lid, 'desconhecido')
            text = f'{name} ({conf:.1f}%)'
        else:
            lid, conf = recognizer.predict(face)
            name = id_labels.get(lid, 'desconhecido')
            text = f'{name} ({conf:.1f})'

        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(rgb)
    plt.axis('off')
    plt.show()

pred_btn.on_click(predict_upload)